# Satellite features extraction

Following up on notebook `02_ground_truth_extraction.ipynb`, this section pulls satellite data from Earth Engine's repositories. This data will later down the pipeline be used to train a machine learning model to predict pollution in the monitoring deserts.


In [1]:
%load_ext autoreload
%autoreload 2

## Imports

In [2]:
from pathlib import Path

import dotenv
import pickle
import gzip
import shutil
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
import yaml
import glob

import ee
import geemap

import monitoring_deserts.data.fetch_earthengine as fetch_ee

## Configuration

Read the configuration parameters from the previous steps and the general project config. Initialize the Earth Engine
client.

In [3]:
# Generic settings
with open("../configs/project_config.yaml") as f:
    gen_config = yaml.safe_load(f)
    
RAW_DATA_DIR = Path(gen_config["data_dir"]["raw"]) / "satellite"
CLEAN_DATA_DIR = Path(gen_config["data_dir"]["clean"]) / "satellite"
PROCESSED_DATA_DIR = Path(gen_config["data_dir"]["processed"]) / "satellite"
MAPS_DIR = Path(gen_config["output_dir"]["maps"]) / "satellite"
PLOTS_DIR = Path(gen_config["output_dir"]["plots"]) / "satellite"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
MAPS_DIR.mkdir(parents=True, exist_ok=True)

# Spacetime settings
with open("../configs/census_scope.yaml") as f:
    census_config = yaml.safe_load(f)

BOUNDING_BOX = tuple(census_config["bounding_box"].values())
DAY_START, DAY_END = census_config["period"]["start"], census_config["period"]["end"]

# Open the ground truth df obtained in notebook 2
gtruth_df = pd.read_parquet(CLEAN_DATA_DIR / "../ground_truth/ground_truth_clean.parquet")

# Initialize client
dotenv.load_dotenv()
ee.Initialize(project="monitoring-deserts")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


## Step 1 - Configuring the request

Build the computation graph for requesting Earth Engine data for tropospheric NO2 and absorbing aerosol index (AAI).

In [4]:
# NO2 image collection
no2_image_collection = (ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2")
         .select("tropospheric_NO2_column_number_density"))
# Aerosol index image collectin
aai_image_collection = (ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_AER_AI")
         .select("absorbing_aerosol_index"))

# Build a feature collection to hold the regions for later reduction
station_fc = fetch_ee.stations_to_feature_collection(gtruth_df, radius_m=2000)

# Build the monthly window where the data gets averaged
m_win = fetch_ee.build_month_windows(DAY_START, DAY_END)

TMP_DATA_DIR = RAW_DATA_DIR / "tmp"
TMP_DATA_DIR.mkdir(parents=True, exist_ok=True)
def _cache_pulled_data(month, df, im_name):
    df.to_parquet(TMP_DATA_DIR / f"tmp_{month}_{im_name}.parquet")

## Step 2 - Pull the data
Very time-consuming step to run locally, so after the first succesful pull just load the .parquet files.

In [7]:
## Pull and save the monthly statistics for NO2 and AAI
#no2_stats = fetch_ee.pull_monthly_stats(no2_image_collection,
#                                        station_fc,
#                                        m_win,
#                                        "no2",
#                                        scale=1000,
#                                        on_month_done=_cache_pulled_data,
#                                       )
#no2_df = pd.concat(no2_stats, ignore_index=True)
#no2_df.to_parquet(RAW_DATA_DIR / "no2_df.parquet")
#aai_stats = fetch_ee.pull_monthly_stats(aai_image_collection,
#                                        station_fc,
#                                        m_win,
#                                        "aai",
#                                        scale=1000,
#                                        on_month_done=_cache_pulled_data,
#                                       )
#aai_df = pd.concat(aai_stats, ignore_index=True)
#aai_df.to_parquet(RAW_DATA_DIR / "aai_df.parquet")
#
## If the stats pulls work as intended, wipe the temporary data directories
#shutil.rmtree(TMP_DATA_DIR, ignore_errors=True)

In [4]:
# Load the dataframes from disk
aai_df = pd.read_parquet(RAW_DATA_DIR / "aai_df.parquet")
no2_df = pd.read_parquet(RAW_DATA_DIR / "no2_df.parquet")

## Step 3 - Data checks and exploration

In [5]:
# Check for missing values
print(f"Missing in NO2: \n{no2_df['mean'].isna().value_counts()}")
print(f"Missing in AAI: \n{aai_df['mean'].isna().value_counts()}")

Missing in NO2: 
mean
False    93364
True       204
Name: count, dtype: int64
Missing in AAI: 
mean
False    93568
Name: count, dtype: int64


In [6]:
# Check whether things are consistent between the previous notebook and this one
print("Number of stations in:")
print(f" Ground truth DataFrame: {gtruth_df['station_id'].nunique()}")
print(f" NO2 DataFrame: {no2_df['station_id'].nunique()}")
print(f" AAI DataFrame: {aai_df['station_id'].nunique()}")
print("Number of analysis months in:")
print(f" Ground truth DataFrame: {gtruth_df['month'].nunique()}")
print(f" NO2 DataFrame: {no2_df['month'].nunique()}")
print(f" AAI DataFrame: {aai_df['month'].nunique()}")

Number of stations in:
 Ground truth DataFrame: 344
 NO2 DataFrame: 344
 AAI DataFrame: 344
Number of analysis months in:
 Ground truth DataFrame: 17
 NO2 DataFrame: 16
 AAI DataFrame: 16


There is one extra month in the ground truth df. This is due to an erroneous interpretation of the half open interval, so that
in this case the last month is `2026-07` and the data for it in gtruth_df has been fully loaded. _I will fix it at the end of the 
project, for now I just drop those rows from the df._

In [12]:
if "2026-07" in gtruth_df["month"].unique():
    gtruth_df = gtruth_df[gtruth_df["month"] != "2026-07"]
    gtruth_df.to_parquet(CLEAN_DATA_DIR / "ground_truth_mod.parquet")
print("Number of analysis months in:")
print(f" Ground truth DataFrame: {gtruth_df['month'].nunique()}")
print(f" NO2 DataFrame: {no2_df['month'].nunique()}")
print(f" AAI DataFrame: {aai_df['month'].nunique()}")

Number of analysis months in:
 Ground truth DataFrame: 16
 NO2 DataFrame: 16
 AAI DataFrame: 16


## Step 4 - Merge the dataframes into a single satellite dataframe

In [19]:
aai_df = aai_df.rename(columns={"station_id" : "station_id",
                                "month" : "month",
                                "mean" : "aai",
                                "stdDev" : "aai_sd"
                               }
                      )[["station_id", "month", "aai", "aai_sd"]]
aai_df["aai_units"] = "dimensionless"
no2_df = no2_df.rename(columns={"station_id" : "station_id",
                                "month" : "month",
                                "mean" : "no2",
                                "stdDev" : "no2_sd"
                               }
                      )[["station_id", "month", "no2", "no2_sd"]]
no2_df["no2_units"] = "mol/m2"
satellite_df = no2_df.merge(aai_df, on=["station_id", "month"])
satellite_df.head()

,station_id,month,no2,no2_sd,no2_units,aai,aai_sd,aai_units
0,50,2025-03-01,0.000106,0.000004,mol/m2,0.208134,0.009537,dimensionless
1,50,2025-03-01,0.000106,0.000004,mol/m2,0.208134,0.009537,dimensionless
2,50,2025-03-01,0.000106,0.000004,mol/m2,0.208134,0.009537,dimensionless
3,50,2025-03-01,0.000106,0.000004,mol/m2,0.208134,0.009537,dimensionless
4,50,2025-03-01,0.000106,0.000004,mol/m2,0.208134,0.009537,dimensionless


In [20]:
satellite_df.to_parquet(CLEAN_DATA_DIR / "satellite_df.parquet")